In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub

In [2]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers
from tensorflow.keras.models import Model

2026-06-26 10:26:49.015287: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782469609.184435      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782469609.238854      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782469609.648604      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782469609.648648      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782469609.648652      23 computation_placer.cc:177] computation placer alr

In [3]:
def find_dataset_path(start_folder):
    for root, dirs, files in os.walk(start_folder):
        if "train" in dirs and "valid" in dirs:
            return root
    return None

base_dir = find_dataset_path("/kaggle/input")

if base_dir is None:
    base_dir = "../input/new-plant-diseases-dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)"
    print("[WARNING] Using fallback path:", base_dir)
else:
    print("[INFO] Dataset found at:", base_dir)

[INFO] Dataset found at: /kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)


In [4]:
DATASET_DIR = base_dir
IMAGE_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 5

In [5]:
train_data = keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

Found 70295 files belonging to 38 classes.


I0000 00:00:1782469647.324220      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782469647.330504      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
val_data = keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "valid"),
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

Found 17572 files belonging to 38 classes.


In [7]:
NUM_CLASSES = len(train_data.class_names)

In [8]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = preprocess_input(inputs)                       # Corrects pixel values for ResNet50
x = layers.RandomFlip("horizontal")(x)             # Simple Data Augmentation
x = layers.RandomRotation(0.05)(x)                 # Simple Data Augmentation
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs, outputs)

In [10]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
print("Starting training...")
model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

Starting training...
Epoch 1/5


I0000 00:00:1782469670.070033      76 cuda_dnn.cc:529] Loaded cuDNN version 91002


550/550 ━━━━━━━━━━━━━━━━━━━━ 347s 607ms/step - accuracy: 0.8386 - loss: 0.5891 - val_accuracy: 0.9579 - val_loss: 0.1572
Epoch 2/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 343s 624ms/step - accuracy: 0.9423 - loss: 0.1962 - val_accuracy: 0.9652 - val_loss: 0.1130
Epoch 3/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 343s 624ms/step - accuracy: 0.9523 - loss: 0.1559 - val_accuracy: 0.9714 - val_loss: 0.0919
Epoch 4/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 344s 626ms/step - accuracy: 0.9582 - loss: 0.1340 - val_accuracy: 0.9738 - val_loss: 0.0828
Epoch 5/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 345s 627ms/step - accuracy: 0.9602 - loss: 0.1231 - val_accuracy: 0.9739 - val_loss: 0.0801


In [12]:
loss, accuracy = model.evaluate(val_data)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

model.save("leaf_disease_resnet50_model.h5")
print("Model saved successfully!")

138/138 ━━━━━━━━━━━━━━━━━━━━ 68s 492ms/step - accuracy: 0.9739 - loss: 0.0801


Validation Accuracy: 97.39%
Model saved successfully!


In [13]:

model.save("leaf_disease_resnet50_model.keras")
print("Model saved successfully in the modern format!")

Model saved successfully in the modern format!
